# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Target Check-in:** Pertemuan 8 (Mulai dikerjakan Pertemuan 6)  
**Nama Mahasiswa:** Muhammad Hilman Al Hazmi  
**NIM:** 202310370311229  
**Username GitHub:** AyamDjago  

---

## Panduan Pengerjaan Milestone 3:
1. **Sumber Data:** Gunakan dataset yang telah dibersihkan pada Milestone 2 (`02_data_cleaning.ipynb`).
2. **Analisis Temporal / Windowing:** Terapkan minimal satu operasi temporal (Polars `group_by_dynamic()`, `rolling_mean()`, `shift()`, atau DuckDB window function `OVER (...)`).
3. **Visualisasi Interaktif:** Bangun minimal 6 visualisasi interaktif informatif menggunakan Plotly Express / Graph Objects atau Altair.
4. **Perumusan Insight:** Tuliskan minimal 5 temuan atau wawasan analitik mendalam berdasarkan hasil visualisasi data di atas.


## Bagian 1: Inisialisasi dan Memuat Data Bersih

Muat dataset hasil pembersihan dari folder `data/` menggunakan Polars.

In [1]:
import math
from pathlib import Path

import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display

pio.renderers.default = "plotly_mimetype"
pio.templates.default = "plotly_white"

NAMA_PARQUET = Path("data/processed/wikipedia_id-bersih.parquet")
folder_aktif = Path.cwd()
path_data = next(
    (folder / NAMA_PARQUET for folder in (folder_aktif, *folder_aktif.parents)
     if (folder / NAMA_PARQUET).is_file()),
    None,
)
if path_data is None:
    raise FileNotFoundError(f"Jalankan Notebook 2 untuk membuat {NAMA_PARQUET}.")

PENGALIHAN = "Pengalihan"
TEKS_KOSONG = "Teks kosong"
LAIN = "Halaman lain"
warna_jenis = {PENGALIHAN: "#9b4f13", TEKS_KOSONG: "#67778a", LAIN: "#176b70"}

# Ambil hanya kolom yang diperlukan. Turunkan panjang asli sebelum melepas kolom teks.
fitur = (
    pl.scan_parquet(path_data)
    .select(
        "id",
        pl.col("timestamp").str.to_datetime(
            format="%Y-%m-%dT%H:%M:%SZ", time_zone="UTC"
        ),
        pl.col("title").str.len_chars().alias("panjang_judul"),
        (pl.col("title") != "tidak diketahui").alias("judul_teramati"),
        pl.col("text").str.len_chars().alias("panjang_asli"),
        pl.when(pl.col("text").str.len_chars() == 0)
        .then(pl.lit(TEKS_KOSONG))
        .when(pl.col("text").str.contains(r"(?i)^\s*#(?:alih|redirect)\b"))
        .then(pl.lit(PENGALIHAN))
        .otherwise(pl.lit(LAIN))
        .alias("jenis"),
    )
    .collect(engine="streaming")
)
assert fitur["timestamp"].null_count() == 0, "Ada timestamp kosong setelah pemuatan."
assert fitur.height > 0, "Parquet hasil cleaning tidak berisi record."

awal = fitur["timestamp"].min()
akhir = fitur["timestamp"].max()
print(f"Jumlah record: {fitur.height:,}")
print(f"Rentang timestamp UTC: {awal:%d-%m-%Y %H:%M:%S} s.d. {akhir:%d-%m-%Y %H:%M:%S}")

Jumlah record: 1,550,872
Rentang timestamp UTC: 19-04-2004 13:00:14 s.d. 20-11-2022 10:49:11


## Bagian 2 — Analisis temporal dan windowing

Pertanyaan analisisnya adalah bagaimana timestamp record tersebar dan bagaimana karakteristik teks berbeda antarjenis halaman. Timestamp menunjukkan waktu yang tercatat pada setiap record dalam dataset. Jumlah per periode dibaca sebagai jumlah record pada dataset dengan timestamp di periode tersebut. Untuk mengukur artikel baru atau semua edit Wikipedia diperlukan riwayat yang sesuai.

Polars mengelompokkan timestamp UTC dengan `group_by_dynamic()` per bulan kalender. Bulan di dalam rentang tanpa record diberi nilai nol. Bulan awal dan akhir diperlakukan sebagai bulan parsial. Rata-rata bergerak tiga bulan dihitung hanya ketika ketiga bulan dalam jendela tercakup penuh.

In [2]:
bulan_awal = awal.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
bulan_akhir = akhir.replace(day=1, hour=0, minute=0, second=0, microsecond=0)

jumlah_per_bulan = (
    fitur.select("timestamp")
    .sort("timestamp")
    .group_by_dynamic("timestamp", every="1mo")
    .agg(pl.len().alias("jumlah"))
    .rename({"timestamp": "bulan"})
)

bulanan = (
    pl.DataFrame({"bulan": pl.datetime_range(bulan_awal, bulan_akhir, "1mo", eager=True)})
    .join(jumlah_per_bulan, on="bulan", how="left")
    .sort("bulan")
    .with_columns(
        pl.col("jumlah").fill_null(0),
        pl.col("bulan").is_in([bulan_awal, bulan_akhir]).alias("parsial"),
    )
    .with_columns(
        pl.when(~pl.col("parsial"))
        .then(pl.col("jumlah"))
        .otherwise(None)
        .rolling_mean(window_size=3)
        .alias("rata_3_bulan")
    )
)
assert bulanan["jumlah"].sum() == fitur.height, "Total agregasi bulanan berbeda."
bulanan.tail(6)

bulan,jumlah,parsial,rata_3_bulan
"datetime[μs, UTC]",u32,bool,f64
2022-06-01 00:00:00 UTC,20199,false,18352.0
2022-07-01 00:00:00 UTC,24794,false,21073.0
2022-08-01 00:00:00 UTC,26329,false,23774.0
2022-09-01 00:00:00 UTC,31319,false,27480.666667
2022-10-01 00:00:00 UTC,40294,false,32647.333333
2022-11-01 00:00:00 UTC,48326,true,null


## Bagian 3 — Enam visualisasi interaktif

Panjang teks dihitung ulang dari `text` sebagai jumlah karakter teks sumber, termasuk markup Wikipedia. Kolom `panjang_teks` dari Notebook 2 sudah melalui clipping pada 3.506 karakter. Pengalihan dideteksi dari awalan `#ALIH` atau `#REDIRECT` tanpa membedakan huruf besar dan kecil serta dengan mengizinkan spasi di awal. Teks dengan panjang nol dikelompokkan sebagai teks kosong. Halaman lain mencakup semua record yang tidak memenuhi dua aturan tersebut.

In [3]:
# 1. Tren waktu dengan range slider dan rata-rata bergerak
parsial = bulanan.filter(pl.col("parsial"))
fig1 = go.Figure()
fig1.add_scatter(
    x=bulanan["bulan"], y=bulanan["jumlah"], mode="lines", name="Record per bulan",
    line={"color": "#176b70"}, hovertemplate="%{x|%b %Y}<br>%{y:,} record",
)
fig1.add_scatter(
    x=bulanan["bulan"], y=bulanan["rata_3_bulan"], mode="lines",
    name="Rata-rata 3 bulan penuh", line={"color": "#58419c", "width": 2},
    hovertemplate="%{x|%b %Y}<br>Rata-rata: %{y:,.0f} record",
)
fig1.add_scatter(
    x=parsial["bulan"], y=parsial["jumlah"], mode="markers", name="Bulan parsial",
    marker={"color": "#9b4f13", "size": 9, "symbol": "diamond"},
    hovertemplate="%{x|%b %Y}<br>%{y:,} record (parsial)",
)
fig1.update_layout(
    title="Jumlah record berdasarkan bulan timestamp",
    xaxis_title="Bulan (UTC)", yaxis_title="Jumlah record", hovermode="x unified",
)
fig1.update_xaxes(rangeslider_visible=True)
fig1.show()

In [4]:
# 2. Histogram panjang asli dengan interval logaritmik
batas_clipping = 3506
positif = fitur.filter(pl.col("panjang_asli") > 0)
histogram = (
    positif.with_columns(
        (pl.col("panjang_asli").log10() * 5).floor().cast(pl.Int32).alias("bin")
    )
    .group_by("bin")
    .agg(pl.len().alias("jumlah"))
)
histogram = (
    pl.DataFrame({"bin": range(histogram["bin"].max() + 1)})
    .join(histogram, on="bin", how="left")
    .with_columns(
        pl.col("jumlah").fill_null(0),
        ((pl.col("bin") + 0.5) / 5).alias("posisi"),
        pl.lit(10).pow(pl.col("bin") / 5).ceil().cast(pl.Int64).alias("dari"),
        (pl.lit(10).pow((pl.col("bin") + 1) / 5).ceil() - 1)
        .cast(pl.Int64).alias("sampai"),
    )
    .filter(pl.col("dari") <= pl.col("sampai"))
)
kosong = fitur.height - positif.height
fig2 = go.Figure(go.Bar(
    x=histogram["posisi"], y=histogram["jumlah"], width=0.185,
    marker_color="#176b70", customdata=histogram.select("dari", "sampai").rows(),
    hovertemplate="%{customdata[0]:,}–%{customdata[1]:,} karakter<br>%{y:,} record<extra></extra>",
))
fig2.add_vline(
    x=math.log10(batas_clipping), line_dash="dash", line_color="#9b4f13",
    annotation_text="Batas clipping Notebook 2",
)
fig2.update_layout(
    title=f"Distribusi panjang teks asli ({kosong:,} teks kosong di luar histogram)",
    yaxis_title="Jumlah record per interval",
)
fig2.update_xaxes(
    title_text="Panjang teks sumber (karakter, skala log)",
    tickvals=list(range(7)), ticktext=["1", "10", "100", "1 rb", "10 rb", "100 rb", "1 jt"],
)
fig2.show()

In [5]:
# 3. Hubungan panjang judul dan teks pada sampel serta korelasi pada populasi analisis
data_korelasi = positif.filter(pl.col("judul_teramati") & (pl.col("panjang_judul") > 0))
spearman = pl.corr("panjang_judul", "panjang_asli", method="spearman")
rho_total = data_korelasi.select(spearman).item()
rho_jenis = (
    data_korelasi.group_by("jenis")
    .agg(spearman.alias("rho"))
    .sort("jenis")
)
sampel = (
    data_korelasi.select("id", "jenis", "panjang_judul", "panjang_asli")
    .sort("id")
    .sample(n=min(4000, data_korelasi.height), seed=42)
)
fig3 = px.scatter(
    sampel, x="panjang_judul", y="panjang_asli", color="jenis", log_y=True,
    color_discrete_map=warna_jenis, opacity=0.42, render_mode="webgl",
    labels={
        "panjang_judul": "Panjang judul (karakter)",
        "panjang_asli": "Panjang teks sumber (karakter)", "jenis": "Jenis halaman",
    },
    title=f"Panjang judul dan panjang teks · Spearman ρ seluruh data analisis = {rho_total:.2f}",
)
fig3.update_traces(marker={"size": 5})
fig3.show()

In [6]:
# 4. Komposisi jenis halaman hasil klasifikasi
komposisi = (
    fitur.group_by("jenis")
    .agg(pl.len().alias("jumlah"), pl.col("panjang_asli").median().alias("median_teks"))
    .with_columns((pl.col("jumlah") / fitur.height * 100).alias("persen"))
    .sort("jumlah", descending=True)
)
fig4 = px.bar(
    komposisi, x="jenis", y="jumlah", color="jenis", color_discrete_map=warna_jenis,
    custom_data=["persen", "median_teks"],
    category_orders={"jenis": komposisi["jenis"].to_list()},
    labels={"jenis": "Jenis halaman", "jumlah": "Jumlah record"},
    title="Komposisi record menurut jenis halaman",
)
fig4.update_traces(
    texttemplate="%{customdata[0]:.2f}%", textposition="outside",
    hovertemplate="%{x}<br>%{y:,} record (%{customdata[0]:.2f}%)"
                  "<br>Median panjang teks: %{customdata[1]:,.0f} karakter<extra></extra>",
    cliponaxis=False,
)
fig4.update_layout(showlegend=False)
fig4.show()

In [7]:
# 5. Heatmap pola bulanan. Sel di luar rentang dibiarkan kosong.
nama_bulan = ["Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]
tahun = list(range(awal.year, akhir.year + 1))
hitung_bulan = {(waktu.year, waktu.month): jumlah for waktu, jumlah in bulanan.select("bulan", "jumlah").iter_rows()}
matriks_jumlah = [[hitung_bulan.get((th, b)) for b in range(1, 13)] for th in tahun]
matriks_warna = [
    [math.log10(1 + jumlah) if jumlah is not None else None for jumlah in baris]
    for baris in matriks_jumlah
]
fig5 = go.Figure(go.Heatmap(
    x=nama_bulan, y=[str(th) for th in tahun], z=matriks_warna,
    customdata=matriks_jumlah, colorscale="Teal", hoverongaps=False,
    colorbar={"title": "log₁₀(1 + jumlah)"},
    hovertemplate="%{x} %{y}<br>%{customdata:,} record<extra></extra>",
))
fig5.add_scatter(
    x=[nama_bulan[waktu.month - 1] for waktu in parsial["bulan"]],
    y=[str(waktu.year) for waktu in parsial["bulan"]],
    customdata=parsial["jumlah"], mode="markers", name="Bulan parsial",
    marker={"symbol": "square-open", "size": 15, "color": "#fb7e28", "line": {"width": 2}},
    hovertemplate="%{x} %{y}<br>%{customdata:,} record (parsial)<extra></extra>",
)
fig5.update_layout(
    title="Pola timestamp menurut tahun dan bulan",
    xaxis_title="Bulan (UTC)", yaxis_title="Tahun",
    yaxis={"type": "category", "categoryorder": "array", "categoryarray": [str(th) for th in tahun]},
    legend={"orientation": "h", "y": -0.16},
)
fig5.show()

In [8]:
# 6. Perbandingan tahun, jenis halaman, dan panjang teks
panjang_tahunan = (
    positif.with_columns(pl.col("timestamp").dt.year().alias("tahun"))
    .group_by("tahun", "jenis")
    .agg(
        pl.col("panjang_asli").median().alias("median_karakter"),
        pl.len().alias("jumlah"),
    )
    .sort("tahun", "jenis")
)
fig6 = px.line(
    panjang_tahunan, x="tahun", y="median_karakter", color="jenis", markers=True,
    color_discrete_map=warna_jenis, custom_data=["jumlah"],
    labels={"tahun": "Tahun (UTC)", "median_karakter": "Median panjang teks sumber (karakter)", "jenis": "Jenis halaman"},
    title="Median panjang teks menurut tahun dan jenis halaman berteks",
)
fig6.update_traces(
    hovertemplate="%{x}<br>Median: %{y:,.0f} karakter<br>%{customdata[0]:,} record",
)
fig6.add_annotation(
    x=1, y=1.07, xref="paper", yref="paper", showarrow=False, xanchor="right",
    text=f"Tahun awal ({awal.year}) dan akhir ({akhir.year}) tercakup sebagian",
)
fig6.update_xaxes(dtick=2)
fig6.show()

## Bagian 4 — Insight analitik

Angka berikut dihitung dari Parquet hasil Milestone 2. Komposisi halaman memakai aturan berbasis awalan teks. Kategori halaman lain mencakup berbagai jenis konten Wikipedia.

In [9]:
# Hitung angka pendukung dari agregasi yang dipakai grafik.
def angka(nilai, desimal=0):
    return f"{nilai:,.{desimal}f}".translate(str.maketrans({",": ".", ".": ","}))


tahun_akhir = akhir.year
bulan_sebanding = bulanan.filter(~pl.col("parsial") & (pl.col("bulan").dt.month() < akhir.month))
per_tahun_sebanding = (
    bulan_sebanding.with_columns(pl.col("bulan").dt.year().alias("tahun"))
    .group_by("tahun")
    .agg(pl.col("jumlah").sum())
)
jumlah_sebanding = dict(per_tahun_sebanding.select("tahun", "jumlah").iter_rows())
jumlah_akhir = jumlah_sebanding[tahun_akhir]
jumlah_sebelumnya = jumlah_sebanding[tahun_akhir - 1]
rasio = jumlah_akhir / jumlah_sebelumnya
jumlah_tahun_akhir = bulanan.filter(pl.col("bulan").dt.year() == tahun_akhir)["jumlah"].sum()
bulan_puncak, jumlah_puncak = (
    bulanan.filter(~pl.col("parsial"))
    .sort(["jumlah", "bulan"], descending=[True, False])
    .select("bulan", "jumlah")
    .row(0)
)
bulan_puncak_sebelumnya, jumlah_puncak_sebelumnya = (
    bulanan.filter(~pl.col("parsial") & (pl.col("bulan").dt.year() == tahun_akhir - 1))
    .sort(["jumlah", "bulan"], descending=[True, False])
    .select("bulan", "jumlah")
    .row(0)
)

stat_panjang = fitur.select(
    pl.col("panjang_asli").median().alias("median"),
    pl.col("panjang_asli").quantile(0.95, interpolation="linear").alias("p95"),
    pl.col("panjang_asli").max().alias("maksimum"),
    (pl.col("panjang_asli") > batas_clipping).sum().alias("di_atas_batas"),
).row(0, named=True)
kategori = {baris["jenis"]: baris for baris in komposisi.iter_rows(named=True)}
rho_per_jenis = dict(rho_jenis.iter_rows())
median_terakhir = {
    baris["jenis"]: baris["median_karakter"]
    for baris in panjang_tahunan.filter(pl.col("tahun") == tahun_akhir).iter_rows(named=True)
}

display(Markdown(f"""
1. **Timestamp terkonsentrasi pada {tahun_akhir}.** Ada **{angka(jumlah_tahun_akhir)} record**
   ({angka(100 * jumlah_tahun_akhir / fitur.height, 1)}% dari dataset) sampai
   {akhir.day} {nama_bulan[akhir.month - 1]}. Pada periode yang sama-sama tercakup penuh
   (Jan–{nama_bulan[akhir.month - 2]}), ada {angka(jumlah_akhir)} record pada {tahun_akhir}
   dan {angka(jumlah_sebelumnya)} pada {tahun_akhir - 1}, atau **{angka(rasio, 2)} kali**.
   Perbandingan ini mengukur sebaran timestamp record dalam dataset. *(Grafik 1 dan 5)*
2. **Sebaran panjang teks memiliki ekor kanan yang panjang.** Median teks sumber
   **{angka(stat_panjang['median'])} karakter**, persentil ke-95 sekitar
   {angka(stat_panjang['p95'])}, dan maksimum {angka(stat_panjang['maksimum'])}.
   Ada {angka(stat_panjang['di_atas_batas'])} record
   ({angka(100 * stat_panjang['di_atas_batas'] / fitur.height, 1)}%)
   di atas batas clipping {angka(batas_clipping)}. Pakai panjang asli untuk menganalisis
   variasi di atas batas itu. *(Grafik 2)*
3. **Hubungan panjang judul dan panjang teks lemah pada data gabungan.** Spearman
   **ρ = {angka(rho_total, 2)}** untuk record dengan judul teramati dan teks berisi.
   Di kelompok pengalihan ρ = {angka(rho_per_jenis[PENGALIHAN], 2)}, sedangkan pada
   halaman lain ρ = {angka(rho_per_jenis[LAIN], 2)}. Karena hubungannya berbeda
   antarjenis, panjang judul perlu ditafsirkan bersama jenis halaman saat membandingkan
   teks. Koefisien dihitung pada seluruh record yang memenuhi filter dan titik yang
   ditampilkan merupakan sampel. *(Grafik 3)*
4. **Pengalihan merupakan bagian besar dari dataset.** Sebanyak
   **{angka(kategori[PENGALIHAN]['jumlah'])} record**
   ({angka(kategori[PENGALIHAN]['persen'], 1)}%) terdeteksi sebagai pengalihan.
   Median panjang teksnya {angka(kategori[PENGALIHAN]['median_teks'])} karakter,
   dibanding {angka(kategori[LAIN]['median_teks'])} karakter pada halaman lain.
   Pada {tahun_akhir}, median kedua kelompok masing-masing
   {angka(median_terakhir[PENGALIHAN])} dan {angka(median_terakhir[LAIN])} karakter.
   Jika tujuan berikutnya adalah analisis isi, laporkan hasil per jenis agar
   komposisi pengalihan terlihat dalam interpretasi. *(Grafik 4 dan 6)*
5. **Puncak bulanan bergeser antara dua tahun terakhir.** Bulan penuh dengan jumlah
   terbesar sepanjang rentang ialah **{nama_bulan[bulan_puncak.month - 1]} {bulan_puncak.year}**
   ({angka(jumlah_puncak)} record). Puncak pada {tahun_akhir - 1} jatuh pada
   {nama_bulan[bulan_puncak_sebelumnya.month - 1]}
   ({angka(jumlah_puncak_sebelumnya)} record). Pergeseran ini memberi alasan untuk
   memeriksa lebih banyak tahun sebelum menyimpulkan adanya pola musiman yang berulang.
   *(Grafik 1 dan 5)*
"""))


1. **Timestamp terkonsentrasi pada 2022.** Ada **489.999 record**
   (31,6% dari dataset) sampai
   20 Nov. Pada periode yang sama-sama tercakup penuh
   (Jan–Okt), ada 441.673 record pada 2022
   dan 222.313 pada 2021, atau **1,99 kali**.
   Perbandingan ini mengukur sebaran timestamp record dalam dataset. *(Grafik 1 dan 5)*
2. **Sebaran panjang teks memiliki ekor kanan yang panjang.** Median teks sumber
   **127 karakter**, persentil ke-95 sekitar
   7.378, dan maksimum 1.887.100.
   Ada 178.602 record
   (11,5%)
   di atas batas clipping 3.506. Pakai panjang asli untuk menganalisis
   variasi di atas batas itu. *(Grafik 2)*
3. **Hubungan panjang judul dan panjang teks lemah pada data gabungan.** Spearman
   **ρ = 0,01** untuk record dengan judul teramati dan teks berisi.
   Di kelompok pengalihan ρ = 0,62, sedangkan pada
   halaman lain ρ = -0,43. Karena hubungannya berbeda
   antarjenis, panjang judul perlu ditafsirkan bersama jenis halaman saat membandingkan
   teks. Koefisien dihitung pada seluruh record yang memenuhi filter dan titik yang
   ditampilkan merupakan sampel. *(Grafik 3)*
4. **Pengalihan merupakan bagian besar dari dataset.** Sebanyak
   **609.517 record**
   (39,3%) terdeteksi sebagai pengalihan.
   Median panjang teksnya 34 karakter,
   dibanding 1.089 karakter pada halaman lain.
   Pada 2022, median kedua kelompok masing-masing
   35 dan 1.205 karakter.
   Jika tujuan berikutnya adalah analisis isi, laporkan hasil per jenis agar
   komposisi pengalihan terlihat dalam interpretasi. *(Grafik 4 dan 6)*
5. **Puncak bulanan bergeser antara dua tahun terakhir.** Bulan penuh dengan jumlah
   terbesar sepanjang rentang ialah **Jan 2022**
   (159.285 record). Puncak pada 2021 jatuh pada
   Jul
   (53.004 record). Pergeseran ini memberi alasan untuk
   memeriksa lebih banyak tahun sebelum menyimpulkan adanya pola musiman yang berulang.
   *(Grafik 1 dan 5)*
